# WorldPop — dependency ratios (archive, live)

`dependency_ratios` ships as per-continent `.7z` archives. The backend resolves the AOI's continent (Africa here), downloads the small archive, extracts the 2010 ratio rasters (total / old-age / young-age) with `py7zr`, and crops them. Needs the `[worldpop]` extra.

In [ ]:
import tempfile

from earthlens.core import EarthLens

d = tempfile.mkdtemp()
paths = EarthLens(
    data_source='worldpop',
    variables=['dependency_ratios'],
    aoi='KEN',
    start='2010',
    end='2010',
    fmt='%Y',
    resolution='1km',
    lat_lim=[-4.0, 4.0],
    lon_lim=[34.0, 41.0],
    path=d,
).download(progress_bar=False)
print('written:', sorted(p.name for p in paths))

Three rasters: total, old-age (65+/working), and young-age (<15/working) dependency. Report the mean total ratio over the AOI:

In [ ]:
import numpy as np
from pyramids.dataset import Dataset

total = [p for p in paths if p.name.endswith('DepRatio_1km.tif')][0]
ds = Dataset.read_file(total)
arr = np.asarray(ds.read_array(), dtype='float64')
nd = ds.no_data_value
nd = nd[0] if isinstance(nd, (tuple, list)) else nd
arr = np.where(arr == nd, np.nan, arr)
print('mean dependency ratio over AOI:', round(float(np.nanmean(arr)), 2))